In [133]:
# =============================================================================
# Configuration — all tunable parameters (edit here only)
# =============================================================================

UNITS_PER_KM = 8.01          # map-unit scale: 8.01 units = 1 km
KNOT1 = 1.5;  KNOT2 = 8.0
FLOOR = 80.0                 # minimum density (people/km²) for seg 3

anchors = [(1.5, 4000), (3.0, 1600), (6.0, 400)]

# Optimiser loss weights
ANCHOR_WEIGHT     = 3.0
FLOOR_WEIGHT      = 2.0
SMOOTH_WEIGHT     = 0.3 
FLOOR_PROBE       = [10.0, 15.0, 25.0, 50.0]
GAMMA1_SEED_RATIO = 0.3
GAMMA3_SEED_RATIO = 0.1
DELTA_SEED        = -0.03    # spatial δ seed (negative → near cluster = higher density)

# DBSCAN cluster detection
EPS_KM      = 1.0            # neighbourhood radius (km)
MIN_SAMPLES = 2


In [134]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt as shapely_wkt
from shapely.strtree import STRtree
from sklearn.cluster import DBSCAN
from scipy.optimize import minimize

df_stage3 = pd.read_csv('csv_input/stage3_synced_polygons.csv')
df_stage3['Area'] = pd.to_numeric(df_stage3['Area'], errors='coerce')
df_stage3 = df_stage3.dropna(subset=['County', 'Area'])

area_analysis = pd.read_csv('area_analysis.csv', usecols=[0, 1, 3])
area_analysis.columns = ['County', 'Expected_Pop', 'Type']
area_analysis['Expected_Pop'] = pd.to_numeric(area_analysis['Expected_Pop'], errors='coerce')
area_analysis = area_analysis.dropna(subset=['County', 'Expected_Pop']).reset_index(drop=True)

print(f'Polygons: {len(df_stage3)}  |  Counties: {len(area_analysis)}')


Polygons: 1709  |  Counties: 35


In [135]:
# Parse centroids, run per-county DBSCAN (weighted centres), compute dist_to_nearest_cluster
gdf = gpd.GeoDataFrame(df_stage3, geometry=df_stage3['geometry'].apply(shapely_wkt.loads))
gdf['cx'] = gdf.geometry.centroid.x
gdf['cy'] = gdf.geometry.centroid.y

eps_units = EPS_KM * UNITS_PER_KM
cluster_results = []

for county in sorted(gdf['County'].unique()):
    grp     = gdf[gdf['County'] == county].copy()
    coords  = grp[['cx', 'cy']].values
    weights = 1.0 / np.maximum(grp['Area'].values, 1e-6)
    labels  = DBSCAN(eps=eps_units, min_samples=MIN_SAMPLES).fit(coords).labels_

    centres = []
    for lbl in sorted(set(labels)):
        if lbl == -1: continue
        mask = labels == lbl
        centres.append({
            'cx': np.average(coords[mask, 0], weights=weights[mask]),
            'cy': np.average(coords[mask, 1], weights=weights[mask]),
        })
    cluster_results.append({
        'county': county, 'gdf': grp.assign(cluster=labels),
        'centres': centres, 'n_clusters': len(centres),
    })

dist_records = []
for r in cluster_results:
    grp = r['gdf']
    if r['n_clusters'] <= 1:
        for _, row in grp.iterrows():
            dist_records.append({'Shape_ID': row['Shape_ID'], 'n_clusters': r['n_clusters'],
                                  'dist_to_nearest_km': 0.0, 'spatial_active': False})
    else:
        cc = np.array([[c['cx'], c['cy']] for c in r['centres']])
        for _, row in grp.iterrows():
            d = np.hypot(cc[:, 0] - row['cx'], cc[:, 1] - row['cy']).min() / UNITS_PER_KM
            dist_records.append({'Shape_ID': row['Shape_ID'], 'n_clusters': r['n_clusters'],
                                  'dist_to_nearest_km': d, 'spatial_active': True})

df_dist = pd.DataFrame(dist_records)
df_stage3 = df_stage3.merge(
    df_dist[['Shape_ID', 'n_clusters', 'dist_to_nearest_km', 'spatial_active']],
    on='Shape_ID', how='left'
)
print(f'DBSCAN  eps={EPS_KM}km  min_samples={MIN_SAMPLES}')
print(f'Spatial-active: {df_stage3["spatial_active"].sum()} polygons  '
      f'| inactive: {(~df_stage3["spatial_active"]).sum()}')


DBSCAN  eps=1.0km  min_samples=2
Spatial-active: 666 polygons  | inactive: 1043


In [136]:
# 3-segment piecewise log-log quadratic, 5 free params [α, β₂, γ₁, γ₂, γ₃]
# β₁, β₃ derived from continuity at KNOT1/KNOT2; all priors read from cell 1.

county_areas = df_stage3.groupby('County')['Area'].apply(np.array).to_dict()

def get_derived_betas(beta2, gamma1, gamma2, gamma3):
    return (beta2 + (gamma2 - gamma1) * np.log(KNOT1),
            beta2 + (gamma2 - gamma3) * np.log(KNOT2))

def density_pw(area, alpha, beta2, gamma1, gamma2, gamma3):
    beta1, beta3 = get_derived_betas(beta2, gamma1, gamma2, gamma3)
    area  = np.atleast_1d(np.asarray(area, dtype=float))
    log_a = np.log(np.maximum(area, 1e-6))
    d     = np.empty_like(area)
    m1, m2, m3 = area < KNOT1, (area >= KNOT1) & (area < KNOT2), area >= KNOT2
    d[m1] = np.exp(alpha + beta1*log_a[m1] + gamma1*log_a[m1]**2)
    d[m2] = np.exp(alpha + beta2*log_a[m2] + gamma2*log_a[m2]**2)
    d[m3] = np.exp(alpha + beta3*log_a[m3] + gamma3*log_a[m3]**2)
    return d

def objective_pw(params):
    alpha, beta2, gamma1, gamma2, gamma3 = params
    pop_loss, n = 0.0, 0
    for _, row in area_analysis.iterrows():
        if row['County'] not in county_areas: continue
        pred      = np.sum(density_pw(county_areas[row['County']], *params) * county_areas[row['County']])
        pop_loss += ((pred - row['Expected_Pop']) / row['Expected_Pop'])**2
        n += 1
    pop_loss /= n
    anchor_loss = sum(
        ((density_pw(np.array([a]), *params)[0] - d) / d)**2 for a, d in anchors
    ) / len(anchors)
    floor_loss = 0.0
    for fa in FLOOR_PROBE:
        d = density_pw(np.array([fa]), *params)[0]
        if d < FLOOR: floor_loss += ((FLOOR - d) / FLOOR)**2
    floor_loss /= len(FLOOR_PROBE)
    return pop_loss + ANCHOR_WEIGHT*anchor_loss + FLOOR_WEIGHT*floor_loss

coeffs = np.polyfit(np.log([a for a, _ in anchors]), np.log([d for _, d in anchors]), 2)
x0 = [coeffs[2], coeffs[1], coeffs[0]*GAMMA1_SEED_RATIO, coeffs[0], coeffs[0]*GAMMA3_SEED_RATIO]

result = minimize(objective_pw, x0, method='Nelder-Mead',
                  options={'maxiter': 30000, 'xatol': 1e-8, 'fatol': 1e-8})

alpha_f, beta2_f, gamma1_f, gamma2_f, gamma3_f = result.x
beta1_f, beta3_f = get_derived_betas(beta2_f, gamma1_f, gamma2_f, gamma3_f)
base_params = [alpha_f, beta2_f, gamma1_f, gamma2_f, gamma3_f]

print(f'Converged: {result.success}  Loss: {result.fun:.6f}')
print(f'α={alpha_f:.4f}  β₁={beta1_f:.4f}  β₂={beta2_f:.4f}  β₃={beta3_f:.4f}')
print(f'γ₁={gamma1_f:.4f}  γ₂={gamma2_f:.4f}  γ₃={gamma3_f:.4f}\n')

anchor_map = dict(anchors)
print(f'{"Area":>10} {"Density":>12} {"Anchor":>10} {"Seg":>4}')
for sz in [0.1, 0.5, 1.0, KNOT1, 3.0, 7.0, KNOT2, 15.0, 30.0]:
    d   = density_pw(np.array([sz]), *base_params)[0]
    seg = 1 if sz < KNOT1 else (2 if sz < KNOT2 else 3)
    ref = f'{anchor_map[sz]:>10,}' if sz in anchor_map else f'{"—":>10}'
    print(f'{sz:>10} {d:>12,.1f} {ref} {seg:>4}{"  ← knot" if sz in (KNOT1, KNOT2) else ""}')


Converged: True  Loss: 0.073777
α=8.4231  β₁=-0.5065  β₂=-0.2343  β₃=-2.1376
γ₁=0.0356  γ₂=-0.6358  γ₃=0.2795

      Area      Density     Anchor  Seg
       0.1     17,640.0          —    1
       0.5      6,576.5          —    1
       1.0      4,551.0          —    1
       1.5      3,727.9      4,000    2  ← knot
       3.0      1,633.3      1,600    2
       7.0        259.8          —    2
       8.0        178.9          —    3  ← knot
      15.0        108.2          —    3
      30.0         80.4          —    3


In [137]:
# Spatial redistribution: boost density near cluster centres, reduce far away,
# preserving each county's total population exactly via per-county rescaling.
# δ (1 param) fitted to minimise log-density jumps across shared polygon edges.

df_stage3.reset_index(drop=True, inplace=True)
shape_id_to_pos = {sid: i for i, sid in enumerate(df_stage3['Shape_ID'].values)}
geoms_list, sids_list = list(gdf.geometry.values), list(gdf['Shape_ID'].values)

tree = STRtree(geoms_list)
adj_pairs_pos = []
for i, geom_i in enumerate(geoms_list):
    for j in tree.query(geom_i):
        if j > i and geom_i.touches(geoms_list[j]):
            pi, pj = shape_id_to_pos.get(sids_list[i]), shape_id_to_pos.get(sids_list[j])
            if pi is not None and pj is not None:
                adj_pairs_pos.append((pi, pj))

adj_i = np.array([p[0] for p in adj_pairs_pos], dtype=int)
adj_j = np.array([p[1] for p in adj_pairs_pos], dtype=int)
print(f'{len(adj_pairs_pos)} adjacent polygon pairs')

all_areas  = df_stage3['Area'].values
all_dists  = df_stage3['dist_to_nearest_km'].values
all_base_d = density_pw(all_areas, *base_params)

active_county_list = sorted(df_stage3[df_stage3['spatial_active']]['County'].unique())
county_idx = {c: grp.index.values for c, grp in df_stage3.groupby('County')}

def apply_spatial(delta):
    final_d = all_base_d.copy()
    for county in active_county_list:
        idx  = county_idx[county]
        base = all_base_d[idx]; area = all_areas[idx]; dist = all_dists[idx]
        raw  = base * np.exp(delta * dist)
        final_d[idx] = raw * (base * area).sum() / (raw * area).sum()
    return final_d

def objective_delta(p):
    log_d = np.log(np.maximum(apply_spatial(p[0]), 1e-6))
    return np.mean((log_d[adj_i] - log_d[adj_j])**2)

result_d      = minimize(objective_delta, [DELTA_SEED], method='Nelder-Mead',
                         options={'maxiter': 5000, 'xatol': 1e-6, 'fatol': 1e-8})
delta_f       = result_d.x[0]
final_density = apply_spatial(delta_f)
print(f'δ = {delta_f:.4f}  ({"near cluster → higher density" if delta_f < 0 else "flat"})')


6324 adjacent polygon pairs
δ = 0.2839  (flat)


In [ ]:
# Type+size weight: area<0.75 → col '0.75'; 0.75≤area<1.5 → col '1.5'; else 1.0
type_wt = pd.read_csv('county_type_weight.csv').set_index('Type')

if 'Type' not in df_stage3.columns:
    df_stage3 = df_stage3.merge(area_analysis[['County', 'Type']], on='County', how='left')

df_stage3['Est_Density'] = final_density.round(1)
df_stage3['Est_Pop']     = (final_density * all_areas).round(0).astype(int)

df_stage3['Weight'] = 1.0
for mask, col in [(df_stage3['Area'] < 0.75, '0.75'),
                  ((df_stage3['Area'] >= 0.75) & (df_stage3['Area'] < 1.5), '1.5')]:
    df_stage3.loc[mask, 'Weight'] = (
        df_stage3.loc[mask, 'Type'].map(type_wt[col]).fillna(1.0).values
    )

df_stage3['Est_Density_W'] = (df_stage3['Est_Density'] * df_stage3['Weight']).round(1)
df_stage3['Est_Pop_W']     = (df_stage3['Est_Density_W'] * df_stage3['Area']).round(0).astype(int)

# County summary
county_cmp = (
    area_analysis[['County', 'Expected_Pop', 'Type']]
    .merge(df_stage3.groupby('County')['Est_Pop'].sum().rename('Base_Pop'),   on='County')
    .merge(df_stage3.groupby('County')['Est_Pop_W'].sum().rename('Wtd_Pop'),  on='County')
)
county_cmp['Base_Err%'] = ((county_cmp['Base_Pop'] - county_cmp['Expected_Pop']) / county_cmp['Expected_Pop'] * 100).round(1)
county_cmp['Wtd_Err%']  = ((county_cmp['Wtd_Pop']  - county_cmp['Expected_Pop']) / county_cmp['Expected_Pop'] * 100).round(1)

active_counties = set(active_county_list)
county_cmp_sorted = county_cmp.sort_values('Base_Err%', key=abs, ascending=False)

print(f"{'County':<6} {'Type':<16} {'Expected':>10} {'Base Pop':>10} {'Base%':>7} {'Wtd Pop':>10} {'Wtd%':>7}  Sp")
print('-' * 76)
for _, r in county_cmp_sorted.iterrows():
    print(f"{r.County:<6} {r.Type:<16} {r.Expected_Pop:>10,.0f} "
          f"{r.Base_Pop:>10,} {r['Base_Err%']:>6.1f}% "
          f"{r.Wtd_Pop:>10,} {r['Wtd_Err%']:>6.1f}%  "
          f"{'★' if r.County in active_counties else ' '}")

print(f"\nBase MAE: {county_cmp['Base_Err%'].abs().mean():.1f}%  "
      f"Wtd MAE: {county_cmp['Wtd_Err%'].abs().mean():.1f}%")

df_stage3[['County', 'Shape_ID', 'Area', 'Type', 'Weight',
           'Est_Density', 'Est_Pop', 'Est_Density_W', 'Est_Pop_W']]


County Type               Expected   Base Pop   Base%    Wtd Pop    Wtd%  Sp
----------------------------------------------------------------------------
W3     Rural - B            47,500     90,242   90.0%     91,289   92.2%  ★
E6     City - D             77,500    114,102   47.2%    116,798   50.7%  ★
E8     City - F             92,500     56,790  -38.6%     64,974  -29.8%   
WL2    Rural - C           185,000    253,755   37.2%    257,143   39.0%  ★
WL1    Rural - C           230,000    312,083   35.7%    318,952   38.7%  ★
E16    City - C            140,000    186,926   33.5%    181,480   29.6%  ★
NC8    Suburb - B          110,000    142,797   29.8%    148,253   34.8%  ★
E19    Suburb - A          180,000    232,885   29.4%    241,714   34.3%  ★
E5     City - B             71,712     52,434  -26.9%     57,464  -19.9%   
E3     City - F            154,302    115,057  -25.4%    131,055  -15.1%   
E10    City - A            170,000    127,997  -24.7%    108,796  -36.0%   
NC7    Cit

,County,Shape_ID,geometry,centroid,Area,n_clusters,dist_to_nearest_km,spatial_active,Type,Est_Density,Est_Pop,Weight,Est_Density_W,Est_Pop_W
0,NC1,NC1_001,"POLYGON ((3279.2822 1418.0581, 3282.2663 1409....",POINT (3282.3270333333335 1413.3738666666668),0.2819,3,1.103417,True,University Town,7702.0,2171,1.2,9242.4,2605
1,NC1,NC1_002,"POLYGON ((3271.8507 1401.4118, 3270.4074 1400,...",POINT (3267.375815355131 1401.8317292319714),0.4191,3,1.629195,True,University Town,7097.9,2975,1.2,8517.5,3570
2,NC1,NC1_003,"POLYGON ((3273.0993 1402.6033, 3271.8507 1401....",POINT (3270.549112312129 1405.4402251671552),0.3371,3,1.054800,True,University Town,6835.5,2304,1.2,8202.6,2765
3,NC1,NC1_004,"POLYGON ((3262.3242 1400, 3267.2913 1405.1833,...",POINT (3263.4816400672717 1404.346460719582),0.4594,3,1.681498,True,University Town,6839.8,3142,1.2,8207.8,3771
4,NC1,NC1_005,"POLYGON ((3276.1418 1410.6351, 3273.9828 1412....",POINT (3273.567316004629 1409.800986759098),0.2069,3,0.444595,True,University Town,7708.7,1595,1.2,9250.4,1914
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1704,WL2,WL2_064,"POLYGON ((3433.6151 860.0658, 3422.6932 859.06...",POINT (3404.488019755355 893.7678475425646),41.0465,4,4.602926,True,Rural - C,64.7,2657,1.0,64.7,2656
1705,WL2,WL2_065,"POLYGON ((3370 960, 3370 944.477, 3370 911.377...",POINT (3390.203119981766 942.3051178182334),26.9803,4,10.896347,True,Rural - C,416.5,11237,1.0,416.5,11237
1706,WL2,WL2_066,"POLYGON ((3370 965.1254, 3370 960, 3387.7333 9...",POINT (3375.291497831056 961.6871218422312),0.5852,4,11.286700,True,Rural - C,33954.2,19870,1.1,37349.6,21857
1707,WL2,WL2_067,"POLYGON ((3410.5864 974.0428, 3408.3109 972.62...",POINT (3413.265886681355 970.7489932701794),1.0565,4,9.017718,True,Rural - C,13084.6,13824,1.0,13084.6,13824


In [151]:
#df_stage3['Final_Density'] = (np.ceil(df_stage3['Est_Density_W'] / 5) * 5).astype(int)

#df_stage3 = (
#     df_stage3[['County', 'Shape_ID', 'geometry', 'Area', 'Weight', 'Final_Density']]
#     .rename(columns={'Weight': 'Final Weight'})
# )
df_stage3
df_stage3.to_csv('stage3_polygon_density_updated.csv', index=False)
